# Stage 1 — FDE / FDA Dispute Detection (Gemma 4)

In [1]:
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


## Configuration

**Edit only this cell.** All paths, model ID, and generation settings are defined here.

In [11]:
# Paths
# Root data folder (must match the os.chdir above)
DATA_FOLDER        = '/content/gdrive/My Drive/WWF_2026/data/'

# Sub-folder containing *_ext.json article files (relative to DATA_FOLDER)
EXTRACTION_FOLDER  = 'n_extracted'

# Sub-folder where output files will be written (relative to DATA_FOLDER)
OUTPUT_FOLDER      = 'n_dispute_detection'

# Output filename (written inside OUTPUT_FOLDER)
OUTPUT_FILENAME    = 'Gemma4_e4b_summary.json'

# Model
MODEL_ID           = 'google/gemma-4-E4B-it'

# Generation
MAX_NEW_TOKENS     = 2048
TEMPERATURE        = 0.1   # near-deterministic; raise to ~0.3 for more variety

# Run control
# Set to a positive int to process only the first N articles (useful for testing).
# Set to None to process all articles.
LIMIT              = None

In [3]:
!pip install -q --upgrade transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 117.6 MB/s eta 0:00:00


In [4]:
import gc
import json
import re
from typing import Dict, List, Optional
import textwrap

import torch
from accelerate import Accelerator
from pydantic import BaseModel, Field, TypeAdapter, ValidationError
from transformers import AutoProcessor, AutoModelForCausalLM
from tqdm.notebook import tqdm

In [5]:
class DetectionSchema(BaseModel):
    """Stage 1 output schema — article-level FDE / FDA detection."""

    contains_fde: bool = Field(
        description="True if the article contains at least one Fisheries Dispute Event (FDE)."
    )
    contains_fda: bool = Field(
        description="True if the article contains at least one Fisheries Dispute Aggregate (FDA)."
    )
    num_candidate_events: int = Field(
        description="Total number of FDE and FDA found in the article."
    )
    confidence: Optional[float] = Field(
        None, ge=0, le=1,
        description="Confidence in the article-level classification (0–1)."
    )
    evidence_summary: str = Field(
        description="Brief evidence-based explanation citing relevant article excerpts."
    )


class OutputFileSchema(BaseModel):
    article_id: str
    detection: List[DetectionSchema]

In [19]:
# JSON extraction helper
import re
import json

def _extract_json_from_text(text: str) -> str:
    """
    Strip markdown fences and extract the JSON string.
    Handles: ```json fences, plain fences, bare JSON,
    and JSON embedded at the end of chain-of-thought output.
    """
    text = text.strip()

    # 1. Try ```json ... ``` or ``` ... ``` fences first
    for pattern in (r"```json(.*?)```", r"```(.*?)```"):
        match = re.search(pattern, text, re.DOTALL)
        if match:
            return match.group(1).strip()

    # 2. Find the LAST occurrence of a JSON object or array.
    #    This handles chain-of-thought models that emit reasoning
    #    before the final JSON answer.
    last_obj   = text.rfind("{")
    last_arr   = text.rfind("[")
    start      = max(last_obj, last_arr)

    if start == -1:
        return text  # nothing found, let the caller handle the error

    # Walk forward to find the matching closing bracket
    open_char  = text[start]
    close_char = "}" if open_char == "{" else "]"
    depth      = 0
    for i in range(start, len(text)):
        if text[i] == open_char:
            depth += 1
        elif text[i] == close_char:
            depth -= 1
            if depth == 0:
                return text[start : i + 1]

    # Unbalanced — return from start to end and let json.loads report the error
    return text[start:]

In [7]:
import os

# Resolve paths from config
os.chdir(DATA_FOLDER)
print('Working directory:', os.getcwd())

extraction_folder_path = os.path.join(DATA_FOLDER, EXTRACTION_FOLDER)
outputs_folder_path    = os.path.join(DATA_FOLDER, OUTPUT_FOLDER)
os.makedirs(outputs_folder_path, exist_ok=True)

extraction_files = sorted(
    f for f in os.listdir(extraction_folder_path) if f.endswith('_ext.json')
)

if LIMIT is not None:
    extraction_files = extraction_files[:LIMIT]
    print(f'LIMIT active — processing first {LIMIT} of {len(extraction_files)} files')

summary_file_path = os.path.join(outputs_folder_path, OUTPUT_FILENAME)

print(f'Extraction folder : {extraction_folder_path}')
print(f'Output folder     : {outputs_folder_path}')
print(f'Output file       : {summary_file_path}')
print(f'Articles to process: {len(extraction_files)}')


Working directory: /content/gdrive/My Drive/WWF_2026/data
Extraction folder : /content/gdrive/My Drive/WWF_2026/data/n_extracted
Output folder     : /content/gdrive/My Drive/WWF_2026/data/n_dispute_detection
Output file       : /content/gdrive/My Drive/WWF_2026/data/n_dispute_detection/Gemma4_e4b_summary.json
Articles to process: 20


## Load model

Run this cell **once**. Re-running it without restarting the runtime will
double-allocate VRAM. If you need to reload, restart the runtime first
(`Runtime → Restart runtime`) so the previous allocation is freed.

In [8]:
def load_model(model_name: str, device: str = "auto"):
    """
    Load a Gemma 4 (or any causal LM) processor and model exactly once.
    Parameters
    model_name : HuggingFace model ID, e.g. 'google/gemma-4-E4B-it'
    device     : 'auto' lets accelerate/device_map decide; pass 'cpu', 'cuda',
                 or 'cuda:0' to pin explicitly.
    Returns
    processor : AutoProcessor  (tokeniser + feature extractor)
    model     : AutoModelForCausalLM  (eval mode, bfloat16, prepared by Accelerator)
    """
    from accelerate import Accelerator
    import torch
    from transformers import AutoProcessor, AutoModelForCausalLM

    accelerator = Accelerator()

    print(f"Loading processor : {model_name}")
    _processor = AutoProcessor.from_pretrained(model_name)

    device_map = "auto" if device == "auto" else device
    dtype      = torch.bfloat16 if torch.cuda.is_available() else torch.float32
    print(f"Loading model     : {model_name}  (dtype={dtype}, device_map={device_map})")
    _model = AutoModelForCausalLM.from_pretrained(
        model_name,
        dtype=dtype,
        device_map=device_map,
        low_cpu_mem_usage=True,
    )
    _model.eval()

    # Capture device map before prepare() wraps the model and loses the attribute.
    _device_map = getattr(_model, 'hf_device_map', None)

    _model = accelerator.prepare(_model)

    print("Model ready.")
    if _device_map:
        print("Device map:", {k: str(v) for k, v in _device_map.items()})
    else:
        print("Device:", next(_model.parameters()).device)
    return _processor, _model


# ── Load once; all downstream cells use `processor` and `model` ──────────────
# WARNING: re-running this cell without restarting the runtime will
# double-allocate VRAM.  Use Runtime → Restart runtime to reload cleanly.
processor, model = load_model(MODEL_ID)


Loading processor : google/gemma-4-E4B-it


processor_config.json:   0%|          | 0.00/1.69k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/17.3k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/5.14k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

... (2 outputs left out)

model.safetensors:   0%|          | 0.00/16.0G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

Model ready.
Device: cuda:0


## Inference Function

In [20]:
# Compile schema once so it isn't re-serialised on every call
_DETECTION_SCHEMA_JSON = json.dumps(TypeAdapter(DetectionSchema).json_schema(), indent=2)

SYSTEM_PROMPT = textwrap.dedent(f'''
You are a fisheries conflict event code engine.
Analyse the supplied article and return a single JSON object.

Definitions:

  Fisheries Dispute Events (FDE):
    An incident in which a fisheries resource is contested, disputed, or the source
    of conflict between a minimum of two actors, at a discrete temporal moment and
    in a discrete location.

  Fisheries Dispute Aggregates (FDA):
    An incident or incidents where a fisheries resource is contested, disputed,
    or the source of conflict between a minimum of two actors, over an uncertain
    or prolonged period of time at a discrete waterbody-district geographic scale.

  Actors:
    human individuals or groups (spontaneous or organized)
    Domestic Fishers, Other, Foreign Fishers, Federal Government, Bandits,
    Military, Police, Regional Government, Resource Security, Local Government.

Task: Analyse the article at the ARTICLE level.

  1. Count total fisheries conflict events → num_candidate_events.
  2. Set contains_fde = true if >=1 FDE is present.
  3. Set contains_fda = true if >=1 FDA is present.
  4. Provide a brief evidence-based summary (≤100 words) citing the strongest evidence.
  5. Return ONLY this JSON object. No preamble, no explanation, no markdown fences.
    {_DETECTION_SCHEMA_JSON}

Rules:
  - Use only information stated in the article.
  - Do not infer missing facts.
  - Output JSON only.
''').strip()


def run_detection(
    extracted_content: Dict,
    processor,
    model,
) -> DetectionSchema | Dict:
    """
    Run Gemma 4 dispute detection on one article dict.

    Parameters:
    extracted_content : dict with at least 'article_text' and 'doc_id' keys
    processor         : AutoProcessor returned by load_model()
    model             : AutoModelForCausalLM returned by load_model()

    Returns a validated DetectionSchema on success, or an error dict on failure.
    """
    article_text = extracted_content.get('article_text', '')
    doc_id       = extracted_content.get('doc_id', 'N/A')

    messages = [
        {"role": "system",  "content": SYSTEM_PROMPT},
        {"role": "user",    "content": f"Article text: {article_text}"},
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True
    )
    inputs = processor(text=text, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[-1]

    # FIX: torch.no_grad() prevents gradient tape accumulation inside the loop,
    #      which was slowly consuming additional VRAM on every article.
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=TEMPERATURE,
            pad_token_id=processor.tokenizer.eos_token_id,
        )

    raw_response = processor.decode(
        output_ids[0][input_len:],
        skip_special_tokens=True,
    ).strip()

    # Parse output
    generated_text = processor.parse_response(raw_response)
    raw_json_string = generated_text.get('content', '')
    extracted_json = _extract_json_from_text(raw_response)

    try:
        detection_data = json.loads(extracted_json)
        return TypeAdapter(DetectionSchema).validate_python(detection_data)
    except json.JSONDecodeError as e:
        print(f"  [JSON ERROR] {doc_id}: {e}\n  Raw prefix: {extracted_json[:300]}")
        return {"error": "JSON_DECODE_ERROR",  "details": str(e), "raw_output": raw_response}
    except ValidationError as e:
        print(f"  [VALIDATION ERROR] {doc_id}: {e}\n  Raw prefix: {extracted_json[:300]}")
        return {"error": "VALIDATION_ERROR",   "details": str(e), "raw_output": raw_response}


## Run detection loop

In [21]:
# summary_file_path is set in the path-setup cell above.

all_dispute_detections = []   # in-memory accumulator for the aggregation step
error_count = 0

for file in tqdm(extraction_files, desc="Detecting disputes"):
    article_id = file.removesuffix('.json').replace('_ext', '')
    file_path  = os.path.join(extraction_folder_path, file)

    with open(file_path, "r", encoding="utf-8") as f:
        extracted_content = json.load(f)
    assert isinstance(extracted_content, dict), f"Expected dict, got {type(extracted_content)}"

    result = run_detection(extracted_content, processor, model)

    if isinstance(result, DetectionSchema):
        entry = OutputFileSchema(article_id=article_id, detection=[result]).model_dump(mode='json')
        status = f"fde={result.contains_fde}  fda={result.contains_fda}  n={result.num_candidate_events}  conf={result.confidence}"
    else:
        entry  = {"article_id": article_id, "detection_error": result}
        status = f"ERROR — {result.get('error')}"
        error_count += 1

    all_dispute_detections.append(entry)
    print(f"  {article_id}: {status}")

    # flush GPU cache after each article to keep VRAM stable.
    torch.cuda.empty_cache()
    gc.collect()

print(f"\nDone. {len(all_dispute_detections)} articles processed, {error_count} errors.")


Detecting disputes:   0%|          | 0/20 [00:00<?, ?it/s]

  nDOC_001: fde=False  fda=False  n=0  conf=1.0
  nDOC_002: fde=False  fda=True  n=1  conf=0.95
  nDOC_003: fde=False  fda=True  n=1  conf=0.95
  nDOC_004: fde=False  fda=True  n=1  conf=0.95
  nDOC_005: fde=False  fda=True  n=1  conf=0.95
  nDOC_006: fde=False  fda=True  n=1  conf=1.0
  nDOC_007: fde=False  fda=True  n=1  conf=0.95
  nDOC_008: fde=False  fda=True  n=1  conf=0.95
  nDOC_009: fde=False  fda=True  n=1  conf=0.95
  nDOC_010: fde=True  fda=True  n=3  conf=0.95
  nDOC_011: fde=False  fda=True  n=1  conf=0.95
  nDOC_012: fde=True  fda=True  n=3  conf=1.0
  nDOC_013: fde=False  fda=True  n=1  conf=1.0
  nDOC_014: fde=False  fda=False  n=0  conf=1.0
  nDOC_015: fde=False  fda=False  n=0  conf=1.0
  nDOC_016: fde=False  fda=False  n=0  conf=1.0
  nDOC_017: fde=False  fda=False  n=0  conf=1.0
  nDOC_018: fde=False  fda=False  n=0  conf=1.0
  nDOC_019: fde=False  fda=False  n=0  conf=1.0
  nDOC_020: fde=False  fda=False  n=0  conf=1.0

Done. 20 articles processed, 0 errors.


In [22]:
result

DetectionSchema(contains_fde=False, contains_fda=False, num_candidate_events=0, confidence=1.0, evidence_summary='The article reports a decline in fish supply and subsequent price increases in Dar es Salaam, caused by strong winds and rough seas that deterred fishermen. However, the text does not describe any conflict, dispute, or contestation between two defined actors over the fisheries resource.')

## Save aggregated output

In [23]:
with open(summary_file_path, 'w', encoding='utf-8') as out:
    json.dump(all_dispute_detections, out, indent=2, ensure_ascii=False)

print(f"Saved {len(all_dispute_detections)} entries → {summary_file_path}")


Saved 20 entries → /content/gdrive/My Drive/WWF_2026/data/n_dispute_detection/Gemma4_e4b_summary.json


## Quick inspection

In [24]:
# Summary statistics
successes  = [e for e in all_dispute_detections if "detection" in e]
errors     = [e for e in all_dispute_detections if "detection_error" in e]
fde_hits   = sum(1 for e in successes if e["detection"][0]["contains_fde"])
fda_hits   = sum(1 for e in successes if e["detection"][0]["contains_fda"])

print(f"Total articles : {len(all_dispute_detections)}")
print(f"Successful     : {len(successes)}")
print(f"Errors         : {len(errors)}")
print(f"contains_fde=T : {fde_hits}")
print(f"contains_fda=T : {fda_hits}")

# Peek at first successful result
if successes:
    print("\nFirst successful result:")
    print(json.dumps(successes[0], indent=2))


Total articles : 20
Successful     : 20
Errors         : 0
contains_fde=T : 2
contains_fda=T : 12

First successful result:
{
  "article_id": "nDOC_001",
  "detection": [
    {
      "contains_fde": false,
      "contains_fda": false,
      "num_candidate_events": 0,
      "confidence": 1.0,
      "evidence_summary": "The article focuses entirely on tourism development potential in Tanzania, discussing beaches, resorts, and marine parks. While deep-sea fishing is mentioned as a potential activity around Mafia Island, there is no evidence of any fisheries resource being contested, disputed, or being the source of conflict between any actors."
    }
  ]
}
